# Stage 4 — Full Graph Construction & Approach B Splitting
### Version 6v2 · Original matching B1 generation logic · Reproducible Colab workflow

**Objective:** Convert completed **Stage 3b outputs + Host / IoT telemetry** into a numerical, graph-structured dataset and prepare the **Approach B** train/validation/test NPZ files used by the three GNN models.

> **Important:** This is the **same executable code** as the earlier *Stage4_Direct_ApproachB_Stage3b_Only_Fixed.ipynb* notebook, whose outputs were checked against the original B1 files. This edition improves the explanations and section organization **without changing the computational cells or splitting rules**.

**Pipeline:** Locate inputs → validate Stage 3b data → create graph nodes → engineer features → create graph edges → validate graph tables → Approach B split → training-only normalization → export graphs & metadata → verify.

**Output root:** `MyDrive/GNN/Stage 4/Approach B/` (under `graph_data/`, `splits/`, and `reports/`).

**Note:** Numerical similarity to a previous run depends on using identical input files and their row ordering, software behavior, and any other relevant randomness. The seed in the existing split code is fixed at 42.

## Step 0 — Libraries and basic helpers

Initialize NumPy/Pandas and the notebook's dependency checker. Run this cell before everything else.

In [ ]:
import sys, os, time
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)

def need(*names):
    """Stop with a clear message if an earlier cell has not been run in this session."""
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(f"Not defined yet: {missing}. Run the cells above first (Runtime -> Run all, "
                           "or run the cells from the top in order).")

print("Libraries loaded successfully")
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

## Step 1 — Connect Google Drive and locate the five input files

The notebook searches below `MyDrive/GNN/` for the original **Host / IoT CSVs** and the three completed **Stage 3b outputs**:

- `scenario_assignments_v6v2.csv` — event-to-scenario membership
- `scenario_summary_v6v2.csv` — scenario metadata
- `links_stage3b_v6v2.csv` — relations between events
- Host telemetry CSV
- IoT network-flow CSV

It **does not require** previously generated Stage 4 A1/B1 NPZ files or `all_scenarios_*` CSV files. Confirm the printed `FOUND` paths are correct; stop if duplicate candidates or missing inputs are reported.

In [ ]:
from pathlib import Path
from collections import defaultdict
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Running outside Colab — searching local directory')
GNN_ROOT = Path('/content/drive/MyDrive/GNN')
ROOTS = [GNN_ROOT] if GNN_ROOT.is_dir() else [Path('/mnt/data'), Path.cwd()]
OUT_ROOT = (GNN_ROOT / 'Stage 4' / 'Approach B') if GNN_ROOT.is_dir() else Path.cwd() / 'Stage4_Direct_B1'
GRAPH_DIR=OUT_ROOT/'graph_data'; SPLIT_DIR=OUT_ROOT/'splits'; REPORT_DIR=OUT_ROOT/'reports'
WANTED = {
    'host':['Host.csv','C1_Output_for_C3_V6_v2_2019.csv'],
    'iot':['IOT_Network_Flow.csv','C2_Output_for_C3_V6_v2_2019.csv'],
    'assign':['scenario_assignments_v6v2.csv'],
    'summary':['scenario_summary_v6v2.csv'],
    'links':['links_stage3b_v6v2.csv'],
}
# Manual override for a duplicate with different contents: PATH_OVERRIDES['host']='/content/drive/.../Host.csv'
PATH_OVERRIDES={}
def locate(names, key):
    if key in PATH_OVERRIDES:
        p=Path(PATH_OVERRIDES[key]); assert p.is_file(),p; return p
    for nm in names:
        found=sorted({p.resolve() for root in ROOTS for p in root.rglob(nm) if OUT_ROOT not in p.parents}, key=str)
        if found:
            if len(found)>1:
                raise RuntimeError(f'Multiple files match {nm}: {found}. Set PATH_OVERRIDES[{key!r}].')
            return found[0]
    return None
PATHS={k:locate(v,k) for k,v in WANTED.items()}
for key,path in PATHS.items():print(f'{key:9s}: {path or "MISSING"}')
missing=[k for k,p in PATHS.items() if p is None]
if missing:raise FileNotFoundError('Missing required inputs: '+', '.join(missing))
protected=[SPLIT_DIR/f'graphs_b1_{k}_v6v2.npz' for k in ('train','validation','test_normal','test_attack')]
protected += [GRAPH_DIR/'final_node_features_v6v2.csv',SPLIT_DIR/'split_b1_v6v2.csv']
existing=[str(p) for p in protected if p.exists()]
if existing:raise FileExistsError('Outputs already exist; preserve them and choose a different OUT_ROOT: '+str(existing))
for p in (GRAPH_DIR,SPLIT_DIR,REPORT_DIR):p.mkdir(parents=True,exist_ok=True)
OUT=GRAPH_DIR
print('Output root:',OUT_ROOT)


## Step 2 — Load and validate Stage 3b + Host/IoT inputs

Read the raw events and Stage 3b tables; parse timestamps, check the input schema and print record counts. The raw event data is **not replaced** with invented records.

In [ ]:
# ==========================================
# STEP 2: LOAD AND CHECK THE INPUTS
# ==========================================
need("PATHS")
FMT = "%d/%m/%Y %H:%M:%S"
host = pd.read_csv(PATHS["host"], dtype=str, keep_default_na=False)
iot = pd.read_csv(PATHS["iot"], dtype=str, keep_default_na=False)
host_ts = pd.to_datetime(host["timestamp"], format=FMT)
iot_ts = pd.to_datetime(iot["timestamp"], format=FMT)
assign = pd.read_csv(PATHS["assign"])
summ = pd.read_csv(PATHS["summary"])
links = pd.read_csv(PATHS["links"])

print("host events :", len(host), "| columns", host.shape[1])
print("iot flows   :", len(iot), "| columns", iot.shape[1])
print("time range  : host", host_ts.min(), "->", host_ts.max(), "| iot", iot_ts.min(), "->", iot_ts.max())
print("assignments :", len(assign), "events | summary:", len(summ), "scenarios | links:", len(links))
print("scenario types:", summ["type"].value_counts().to_dict())
print("\nchecks:")
print("  assignments cover every event once :", len(assign) == len(host) + len(iot) and assign["node"].is_unique)
print("  summary scenarios have >= 2 events :", bool((summ["n_events"] >= 2).all()))
print("  12,780 scenarios / 101,695 events  :", len(summ) == 12780 and int(summ["n_events"].sum()) == 101695)

## Step 3 — Build graph nodes and engineer features

Each node represents a scenario event. Features capture the node type, host-process information, IoT-flow properties, and other engineered numerical signals. All scenario graph features are prepared before bucket export.

### Step 3A — Assemble the scenario node table

Create stable node indices and attach timestamps, scenario IDs and source event references.

In [ ]:
# ==========================================
# STEP 3a: NODE TABLE (events inside the 12,780 scenarios, ordered by scenario and time)
# ==========================================
need("host", "iot", "assign", "summ", "host_ts", "iot_ts")
managed_ips = set(host["host_ip"])
inc = set(summ["scenario_id"])
nodes = assign[assign["scenario_id"].isin(inc)].copy()
nodes["kind"] = nodes["node"].str.split(":").str[0]
nodes["row"] = nodes["node"].str.split(":").str[1].astype(int)

h_ts = host_ts.astype("datetime64[s]").astype("int64").to_numpy()
i_ts = iot_ts.astype("datetime64[s]").astype("int64").to_numpy()
iot_host_ip = np.where(iot["src_ip"].isin(managed_ips), iot["src_ip"],
                       np.where(iot["dst_ip"].isin(managed_ips), iot["dst_ip"], ""))
kind, rows = nodes["kind"].to_numpy(), nodes["row"].to_numpy()
isH = kind == "host"
nodes["ts_s"] = np.where(isH, h_ts[np.where(isH, rows, 0)], i_ts[np.where(~isH, rows, 0)])
nodes["host_ip"] = np.where(isH, host["host_ip"].to_numpy(object)[np.where(isH, rows, 0)],
                            iot_host_ip[np.where(~isH, rows, 0)])
nodes["label"] = np.where(isH, host["label"].astype(int).to_numpy()[np.where(isH, rows, 0)],
                          iot["label"].astype(int).to_numpy()[np.where(~isH, rows, 0)])

nodes = nodes.sort_values(["scenario_id", "ts_s", "kind", "row"], kind="stable").reset_index(drop=True)
nodes["node_id"] = np.arange(len(nodes))
g = nodes.groupby("scenario_id")["ts_s"]
nodes["since_start"] = nodes["ts_s"] - g.transform("min")
nodes["since_prev"] = g.diff().fillna(0)

kind, rows = nodes["kind"].to_numpy(), nodes["row"].to_numpy()
print("nodes:", len(nodes), "| host", int((kind == "host").sum()), "| iot", int((kind == "iot").sum()),
      "| scenarios:", nodes["scenario_id"].nunique())

### Step 3B — Engineer numerical node features

Create one-hot categorical indicators, numerical transforms, node-kind indicators and relative-time features. Later we fit the continuous-feature scaling parameters **only on training graphs**.

In [ ]:
# ==========================================
# STEP 3b: NODE FEATURES (all numbers, not scaled)
# ==========================================
need("nodes", "host", "iot", "kind", "rows")
feat, group, to_scale = {}, {}, []

def add(name, arr, grp, scale=False):
    assert name not in feat, name
    feat[name] = arr
    group[name] = grp
    if scale:
        to_scale.append(name)

def onehot(kind_name, frame, col, prefix, grp):
    m = kind == kind_name
    vals = np.full(len(nodes), None, dtype=object)
    vals[m] = frame[col].to_numpy(object)[rows[m]]
    for v in sorted(frame[col].unique()):
        label = "none" if v in ("-", "") else v
        name = f"{prefix}{col}={label}"
        add(name, (vals == v).astype(np.int8), grp)

def numeric(kind_name, frame, col, prefix, grp, log):
    m = kind == kind_name
    x = np.zeros(len(nodes))
    x[m] = pd.to_numeric(frame[col]).to_numpy(float)[rows[m]]
    assert x.min() >= 0 or not log, col
    add(f"{prefix}{col}" + ("_log1p" if log else ""), np.log1p(x) if log else x, grp, scale=True)

add("is_host", (kind == "host").astype(np.int8), "kind")
add("is_iot", (kind == "iot").astype(np.int8), "kind")
for c in ["event_type", "process_name", "protocol", "syscall_name", "syscall_result", "process_state", "file_operation", "trace_source"]:
    onehot("host", host, c, "h_", "host_categorical")
for c, lg in [("memory_alloc_bytes", True), ("disk_read_bytes", True), ("disk_write_bytes", True), ("cpu_usage", False)]:
    numeric("host", host, c, "h_", "host_numeric", lg)
for c in ["service", "protocol"]:
    onehot("iot", iot, c, "i_", "iot_categorical")
for c in ["flow_duration", "src_bytes", "dst_bytes", "src_packets", "dst_packets", "iat_mean", "iat_std",
          "packet_size_mean", "packet_size_std", "connection_rate"]:
    numeric("iot", iot, c, "i_", "iot_numeric", True)
add("t_since_start_log1p", np.log1p(nodes["since_start"].to_numpy(float)), "time", scale=True)
add("t_since_prev_log1p", np.log1p(nodes["since_prev"].to_numpy(float)), "time", scale=True)
for c in ["trace_mode", "threat_state"]:
    onehot("host", host, c, "c1_", "component")
numeric("host", host, "threat_confidence_score", "c1_", "component", False)
onehot("iot", iot, "predicted_class", "c2_", "component")
numeric("iot", iot, "confidence_score", "c2_", "component", False)

X = pd.DataFrame(feat)
FEATURE_GROUPS = pd.Series(group)
print("node feature matrix:", X.shape, "| columns to scale:", len(to_scale))
display(FEATURE_GROUPS.value_counts().to_frame("features"))

## Step 4 — Construct graph edges

### Step 3C in the original algorithm — Stage 3b links → graph edges

Resolve node endpoints, retain within-scenario links and construct edge attributes (including relation type and time/proximity information). This code is unchanged from the verified matching version.

In [ ]:
# ==========================================
# STEP 3c: EDGES (linked pairs inside one scenario)
# ==========================================
need("nodes", "links", "X")
nid = pd.Series(nodes["node_id"].to_numpy(), index=nodes["node"].to_numpy())
scn = pd.Series(nodes["scenario_id"].to_numpy(), index=nodes["node"].to_numpy())
l = links[links["node_a"].isin(nid.index) & links["node_b"].isin(nid.index)].copy()
l["a"], l["b"] = nid.reindex(l["node_a"]).to_numpy(), nid.reindex(l["node_b"]).to_numpy()
l["sa"], l["sb"] = scn.reindex(l["node_a"]).to_numpy(), scn.reindex(l["node_b"]).to_numpy()
print("links between events of the 12,780 scenarios:", len(l), "| of these inside one scenario:", int((l["sa"] == l["sb"]).sum()))
l = l[l["sa"] == l["sb"]].copy()
l["lo"], l["hi"] = np.minimum(l["a"], l["b"]), np.maximum(l["a"], l["b"])

pair = l.groupby(["lo", "hi"]).agg(scenario_id=("sa", "first"), points=("points", "sum"), gap=("gap_sec", "min"))
types = pd.crosstab([l["lo"], l["hi"]], l["link_type"]).gt(0).astype(int)
types.columns = ["e_type=" + c for c in types.columns]
edges = pair.join(types).reset_index().sort_values(["scenario_id", "lo", "hi"], kind="stable").reset_index(drop=True)

ts_n, hip_n = nodes["ts_s"].to_numpy(), nodes["host_ip"].to_numpy(object)
gap_s = np.abs(ts_n[edges["hi"]] - ts_n[edges["lo"]])
same_host = ((hip_n[edges["lo"]] == hip_n[edges["hi"]]) & (hip_n[edges["lo"]] != "")).astype(int)
short_gap = (gap_s < 300).astype(int)
edges["e_score"] = edges["points"] + same_host + short_gap
edges["e_log_gap"] = np.log1p(edges["gap"])
edges["e_same_host"], edges["e_short_gap"] = same_host, short_gap
EDGE_TYPE_COLS = [c for c in edges.columns if c.startswith("e_type=")]
EDGE_COLS = ["e_score", "e_log_gap", "e_same_host", "e_short_gap"] + EDGE_TYPE_COLS
EDGE_SCALE = ["e_score", "e_log_gap"]
print("edges (undirected pairs):", len(edges), "| edge features:", len(EDGE_COLS))
display(edges[EDGE_COLS].describe().T[["mean", "min", "max"]].round(2).head(8))
print("scenarios without any edge:", len(set(nodes["scenario_id"]) - set(edges["scenario_id"])))

## Step 5 — Validate and export the finalized graph tables

Confirm numerical features, valid node/edge indices, non-missing values and scenario consistency. Export common **graph CSVs** to `graph_data/` before splitting:

- `final_node_features_v6v2.csv`
- `final_edges_v6v2.csv`
- `final_node_labels_v6v2.csv`
- `final_scenario_info_v6v2.csv`
- `feature_names_v6v2.csv`

These CSVs describe the shared constructed graph dataset; the four NPZ files later contain the specific model-ready splits.

In [ ]:
# ==========================================
# STEP 4: CHECK AND SAVE THE FINALIZED DATASET
# ==========================================
need("X", "nodes", "edges", "summ", "FEATURE_GROUPS", "EDGE_COLS", "OUT")
assert all(pd.api.types.is_numeric_dtype(X[c]) for c in X.columns), "a feature column is not numeric"
assert not X.isna().any().any() and not edges[EDGE_COLS].isna().any().any(), "NaN found"
assert not any(w in c.lower() for c in X.columns for w in ("label", "attack_type", "scenario")), "label-like column in the features"
sizes = nodes.groupby("scenario_id").size()
assert (sizes >= 2).all(), "a scenario has fewer than 2 nodes"
assert set(edges["scenario_id"]) == set(nodes["scenario_id"]), "a scenario has no edge"
sc_of_node = nodes["scenario_id"].to_numpy()
assert (sc_of_node[edges["lo"]] == edges["scenario_id"].to_numpy()).all() and (sc_of_node[edges["hi"]] == edges["scenario_id"].to_numpy()).all()
assert (sizes.reindex(summ["scenario_id"]).to_numpy() == summ["n_events"].to_numpy()).all(), "scenario sizes differ from Stage 3b"
print("all features numeric            :", True)
print("no missing values               :", True)
print("no label-like feature column    :", True)
print("every scenario >= 2 nodes, >= 1 edge, edges inside one scenario, sizes equal Stage 3b :", True)
print("features:", X.shape[1], "| groups:", FEATURE_GROUPS.value_counts().to_dict())

fin = pd.concat([nodes[["node_id", "node", "scenario_id"]].reset_index(drop=True), X.round(6)], axis=1)
fin.to_csv(OUT / "final_node_features_v6v2.csv", index=False)
edges[["scenario_id", "lo", "hi"] + EDGE_COLS].rename(columns={"lo": "node_a_id", "hi": "node_b_id"}).round(6).to_csv(
    OUT / "final_edges_v6v2.csv", index=False)
summ[["scenario_id", "type", "n_events", "n_label1", "label1_share"]].to_csv(OUT / "final_scenario_info_v6v2.csv", index=False)
nodes[["node_id", "node", "scenario_id", "label"]].to_csv(OUT / "final_node_labels_v6v2.csv", index=False)
pd.DataFrame({"feature": list(X.columns), "group": [FEATURE_GROUPS[c] for c in X.columns],
              "scaled_later": [c in to_scale for c in X.columns]}).to_csv(OUT / "feature_names_v6v2.csv", index=False)
for f in ["final_node_features_v6v2.csv", "final_edges_v6v2.csv", "final_scenario_info_v6v2.csv", "final_node_labels_v6v2.csv", "feature_names_v6v2.csv"]:
    print("saved", OUT / f)

## Step 6 — Approach B split design and scenario metadata

**Keep the original, previously matching allocation algorithm unchanged.** Training is normal-only. Both validation and test contain normal and attack scenarios.

| Split | Normal | Attack | Total |
|---|---:|---:|---:|
| Train | 8,946 | 0 | 8,946 |
| Validation | 722 | 556 | 1,278 |
| Test | 1,112 | 1,444 | 2,556 |
| **Total** | **10,780** | **2,000** | **12,780** |

The next code cell initializes seed 42, fixed target counts, training IDs and scenario metadata. **70:10:20 refers to all 12,780 scenarios**, not a separately stratified 70:10:20 ratio within the 2,000 attacks.

In [ ]:
SEED=42
GROUP_GAP_S=60
TARGET={'train':(8946,0),'validation':(722,556),'test':(1112,1444)}
SIZE_BANDS=[('2-4',2,4),('5-8',5,8),('9-16',9,16),('17+',17,10**9)]
def band(n):return next(name for name,lo,hi in SIZE_BANDS if lo<=n<=hi)
normal_ids=summ.loc[summ['type']=='normal','scenario_id'].to_numpy()
attack_ids=summ.loc[summ['type']!='normal','scenario_id'].to_numpy()
assert len(summ)==12780 and len(normal_ids)==10780 and len(attack_ids)==2000
train_ids=np.random.default_rng(SEED).permutation(normal_ids)[:8946]
train_set=set(map(int,train_ids))

# Derive scenario-level splitting metadata directly from Stage 3b nodes/links and raw Host/IoT logs.
# Do NOT load old Stage 4 all_scenarios_overview / all_scenarios_events CSVs.
base=summ.set_index('scenario_id').copy()
meta=pd.DataFrame(index=base.index)
meta['scenario_type']=np.where(base['type'].eq('normal'),'normal','attack-containing')
meta['n_events_total']=base['n_events'].astype(int)
meta['is_attack']=base['type'].ne('normal').astype(int)
meta['size_band']=meta['n_events_total'].map(band)

# Host/IP/time info comes from actual scenario-assigned events prepared above.
agg=nodes.groupby('scenario_id').agg(t0=('ts_s','min'),t1=('ts_s','max'))
meta['t0']=agg['t0']-int(nodes['ts_s'].min())
meta['t1']=agg['t1']-int(nodes['ts_s'].min())
meta['hs']=nodes.groupby('scenario_id')['host_ip'].agg(lambda s:frozenset(str(x) for x in s if pd.notna(x) and str(x)!=''))
meta['hosts']=meta['hs'].map(lambda s:';'.join(sorted(s)))

# Cross-domain evidence from Stage 3b links that remain inside the same scenario.
linked=l.copy()
linked['cross_domain']=(linked['node_a'].astype(str).str.split(':').str[0]
                         !=linked['node_b'].astype(str).str.split(':').str[0])
xdom=linked.groupby('sa')['cross_domain'].any()
meta['has_cross_domain_link']=xdom.reindex(meta.index,fill_value=False).astype(int)

# Attack-type metadata is for validation/test stratification only, never a GNN input.
node_type=nodes[['scenario_id','kind','row','label']].copy()
node_type['raw_type']=''
hmask=node_type['kind'].eq('host').to_numpy()
imask=node_type['kind'].eq('iot').to_numpy()
node_type.loc[hmask,'raw_type']=host['type'].astype(str).to_numpy()[node_type.loc[hmask,'row'].to_numpy(int)]
node_type.loc[imask,'raw_type']=iot['attack_type'].astype(str).to_numpy()[node_type.loc[imask,'row'].to_numpy(int)]
att=node_type[node_type['label'].astype(int).eq(1)]
def main_attack_type(s):
    counts=s.value_counts()
    if counts.empty:return ''
    winners=counts[counts.eq(counts.max())].index.astype(str)
    return sorted(winners)[0]
main_type=att.groupby('scenario_id')['raw_type'].agg(main_attack_type)
all_types=att.groupby('scenario_id')['raw_type'].agg(lambda s:'|'.join(sorted(set(map(str,s)))))
meta['attack_type']=main_type.reindex(meta.index).fillna('')
meta['attack_types_all']=all_types.reindex(meta.index).fillna('')
assert set(meta.index)==set(summ['scenario_id']) and int(meta['is_attack'].sum())==2000
assert (meta.loc[list(train_set),'is_attack']==0).all()
assert not meta[['t0','t1','n_events_total']].isna().any().any()
print('Stage 3b metadata created without old Stage 4 CSVs.')
print('Training normal graphs:',len(train_set),'| Held-out normal:',len(normal_ids)-len(train_set),'| Held-out attacks:',len(attack_ids))


### Step 6B — Form proximity groups for the held-out scenarios

The existing algorithm groups temporally nearby scenarios sharing host context, supporting the held-out validation/test assignment procedure. This is not a guarantee that all event-source leakage has been eliminated; that remains part of the separate research audit.

In [ ]:
def proximity_groups(df, gap):
    """Connected components: shared host AND time windows overlapping or <= gap seconds apart (same-class scenarios)."""
    n = len(df)
    t0, t1, hs = df["t0"].to_numpy(), df["t1"].to_numpy(), df["hs"].to_numpy()
    parent = list(range(n))

    def find_root(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for a in range(n):
        for b in np.where((t0 <= t1[a] + gap) & (t1 >= t0[a] - gap))[0]:
            if b > a and (hs[a] & hs[b]):
                ra, rb = find_root(a), find_root(int(b))
                if ra != rb:
                    parent[ra] = rb
    roots = [find_root(i) for i in range(n)]
    first = {}
    for i, r in enumerate(roots):                       # group id = smallest scenario id of the group
        first[r] = min(first.get(r, 10 ** 12), int(df.index[i]))
    return pd.Series([first[r] for r in roots], index=df.index)

def allocate(df, n_val, rng):
    """Group-aware stratified choice of exactly n_val scenarios for validation. df: one class; needs group, stratum."""
    grp = df.groupby("group").agg(size=("group", "size"), stratum=("stratum", "first")).reset_index()
    total = len(df)
    strata = df["stratum"].value_counts()
    ideal = strata * n_val / total
    quota = np.floor(ideal).astype(int)
    rest = n_val - int(quota.sum())
    for s in (ideal - quota).sort_values(ascending=False, kind="stable").index[:rest]:
        quota[s] += 1
    chosen, got = set(), {s: 0 for s in strata.index}
    for s in strata.index:
        g = grp[grp["stratum"] == s].sort_values("group")
        order = rng.permutation(len(g))
        for k in order:
            gid, size = int(g.iloc[k]["group"]), int(g.iloc[k]["size"])
            if got[s] + size <= quota[s]:
                chosen.add(gid)
                got[s] += size
    # exact total: move whole SINGLETON groups (never break a group) from the most over- / under-filled strata
    cur = sum(int(grp.loc[grp["group"] == g, "size"].iloc[0]) for g in chosen)
    single = grp[grp["size"] == 1]
    while cur != n_val:
        want_more = cur < n_val
        pool = single[~single["group"].isin(chosen)] if want_more else single[single["group"].isin(chosen)]
        if pool.empty:
            raise RuntimeError(f"cannot reach exactly {n_val} validation scenarios without breaking a group (at {cur})")
        gap = {s: (ideal[s] - got[s]) for s in strata.index}
        pool = pool.assign(score=pool["stratum"].map(gap) * (1 if want_more else -1))
        top = pool[pool["score"] == pool["score"].max()]
        gid = int(top.iloc[int(rng.integers(len(top)))]["group"])
        s = top[top["group"] == gid]["stratum"].iloc[0]
        chosen.add(gid) if want_more else chosen.discard(gid)
        got[s] += 1 if want_more else -1
        cur += 1 if want_more else -1
    return df["group"].isin(chosen)
print("Group-aware allocation helpers ready")

### Step 6C — Allocate validation and test scenarios

Reuse the exact seed, stratification and assignment rules from the matching notebook. The result should produce four logical buckets: `train`, `validation`, `test_normal`, and `test_attack`. Do **not** edit the random generator or sorting logic simply to improve presentation.

In [ ]:
pool=meta.loc[~meta.index.isin(train_set)].copy()
pool['group']=-1;pool['stratum']=''
for cls,df in pool.groupby('is_attack'):
    g=proximity_groups(df,GROUP_GAP_S)
    pool.loc[df.index,'group']=g
    if cls==1:
        pool.loc[df.index,'stratum']=df['attack_type']+'|'+df['size_band']
    else:
        pool.loc[df.index,'stratum']=df['size_band']+'|xdom='+df['has_cross_domain_link'].astype(str)
assert (pool['group']>=0).all()
gsz=pool.groupby('group').size()
pool['group_size']=pool['group'].map(gsz)
rng=np.random.default_rng(SEED)
pool['split']='test'
for cls,n_val in ((0,722),(1,556)):
    df=pool[pool.is_attack==cls]
    selected=allocate(df,n_val,rng)
    pool.loc[df.index[selected],'split']='validation'
meta['split']='train'
meta.loc[pool.index,'split']=pool['split']
meta['group']=pool['group'];meta['group_size']=pool['group_size'];meta['stratum']=pool['stratum']
counts=meta.groupby(['split','is_attack']).size().unstack(fill_value=0).reindex(columns=[0,1],fill_value=0)
counts.columns=['normal','attack'];counts['total']=counts.sum(axis=1)
print(counts.to_string())
for split,(normal,attack) in TARGET.items():
    assert int(counts.loc[split,'normal'])==normal and int(counts.loc[split,'attack'])==attack
assert (pool.groupby('group')['split'].nunique()<=1).all(),'Related held-out groups split across validation and testing'
buckets={
 'train':np.sort(np.array(list(train_set),dtype=np.int64)),
 'validation':np.sort(meta.index[meta['split']=='validation'].to_numpy(dtype=np.int64)),
 'test_normal':np.sort(meta.index[(meta['split']=='test')&(meta['is_attack']==0)].to_numpy(dtype=np.int64)),
 'test_attack':np.sort(meta.index[(meta['split']=='test')&(meta['is_attack']==1)].to_numpy(dtype=np.int64)),
}
assert len(set().union(*(set(map(int,x)) for x in buckets.values())))==12780
assert sum(map(len,buckets.values()))==12780
print('Direct B1 split selection PASS')


## Step 7 — Scale features using training data only

Fit the numerical node- and edge-scaling statistics on the selected **training scenarios**. Apply those same statistics to validation/test. The training-only scaler values are also exported to CSV.

This prevents validation/test statistics from influencing normalization parameters.

In [ ]:
# ==========================================
# STEP 6: SCALING (fitted on the training scenarios only)
# ==========================================
need("X", "nodes", "edges", "buckets", "to_scale", "EDGE_SCALE", "SPLIT_DIR")
train_set = set(buckets["train"])
node_train = nodes["scenario_id"].isin(train_set).to_numpy()
edge_train = edges["scenario_id"].isin(train_set).to_numpy()

stats = []
Xs = X.copy().astype(np.float64)
for c in to_scale:
    mu, sd = X.loc[node_train, c].mean(), X.loc[node_train, c].std(ddof=0)
    sd = sd if sd > 0 else 1.0
    Xs[c] = (X[c] - mu) / sd
    stats.append(("node", c, mu, sd))
Es = edges[EDGE_COLS].copy().astype(np.float64)
for c in EDGE_SCALE:
    mu, sd = edges.loc[edge_train, c].mean(), edges.loc[edge_train, c].std(ddof=0)
    sd = sd if sd > 0 else 1.0
    Es[c] = (edges[c] - mu) / sd
    stats.append(("edge", c, mu, sd))
scaler = pd.DataFrame(stats, columns=["table", "column", "mean_train", "std_train"])
scaler.to_csv(SPLIT_DIR / "scaler_b1_v6v2.csv", index=False)
print("scaled columns:", len(to_scale), "node +", len(EDGE_SCALE), "edge | statistics from", int(node_train.sum()), "training nodes and", int(edge_train.sum()), "training edges")
print("training nodes after scaling: mean of scaled columns = %.3f, std = %.3f (should be ~0 and ~1)" % (
    Xs.loc[node_train, to_scale].mean().abs().max(), Xs.loc[node_train, to_scale].std(ddof=0).mean()))
print("no NaN / inf after scaling:", bool(np.isfinite(Xs.to_numpy()).all() and np.isfinite(Es.to_numpy()).all()))
print("saved", SPLIT_DIR / "scaler_b1_v6v2.csv")
display(scaler.head(6).round(3))

## Step 8 — Build and save four model-ready B1 graph NPZ files

The NPZ arrays describe scenario graph boundaries, node features, graph connectivity and edge attributes. Output names:

| File | Graphs |
|---|---:|
| `graphs_b1_train_v6v2.npz` | 8,946 normal |
| `graphs_b1_validation_v6v2.npz` | 722 normal + 556 attack |
| `graphs_b1_test_normal_v6v2.npz` | 1,112 normal |
| `graphs_b1_test_attack_v6v2.npz` | 1,444 attack |

These go to the `splits/` subfolder. Existing save behavior is preserved. Before running, make sure you have a backup if you previously generated data in that location.

In [ ]:
# ==========================================
# STEP 7a: BUILD AND SAVE THE BUCKET FILES
# ==========================================
need("Xs", "Es", "nodes", "edges", "buckets", "summ", "FEATURE_GROUPS", "EDGE_COLS", "SPLIT_DIR")
nstart = nodes.groupby("scenario_id")["node_id"].min()
ncount = nodes.groupby("scenario_id").size()
estart = edges.reset_index().groupby("scenario_id")["index"].min()
ecount = edges.groupby("scenario_id").size()
is_attack = (summ.set_index("scenario_id")["type"] != "normal").astype(np.int8)
xs = Xs.to_numpy(np.float32)
es = Es.to_numpy(np.float32)
lo_all, hi_all = edges["lo"].to_numpy(), edges["hi"].to_numpy()
node_label_all = nodes["label"].to_numpy(np.int8)
feature_names = np.array([str(c) for c in Xs.columns], dtype="U80")
feature_groups = np.array([str(FEATURE_GROUPS[c]) for c in Xs.columns], dtype="U40")

summary_rows = []
for bname in ("train", "validation", "test_normal", "test_attack"):
    ids = np.sort(buckets[bname])
    nidx = np.concatenate([np.arange(nstart[s], nstart[s] + ncount[s]) for s in ids])
    node_ptr = np.r_[0, np.cumsum([ncount[s] for s in ids])].astype(np.int64)
    ei, ea, eptr = [], [], [0]
    for s in ids:
        sl = slice(estart[s], estart[s] + ecount[s])
        lo, hi = lo_all[sl] - nstart[s], hi_all[sl] - nstart[s]
        ei.append(np.concatenate([np.stack([lo, hi]), np.stack([hi, lo])], axis=1))
        ea.append(np.concatenate([es[sl], es[sl]], axis=0))
        eptr.append(eptr[-1] + 2 * (ecount[s]))
    np.savez_compressed(SPLIT_DIR / f"graphs_b1_{bname}_v6v2.npz",
                        x=xs[nidx], node_ptr=node_ptr, edge_index=np.concatenate(ei, axis=1).astype(np.int64),
                        edge_attr=np.concatenate(ea, axis=0), edge_ptr=np.array(eptr, dtype=np.int64),
                        scenario_id=ids.astype(np.int64), scenario_is_attack=is_attack.reindex(ids).to_numpy(),
                        node_label=node_label_all[nidx], feature_names=feature_names, feature_groups=feature_groups,
                        edge_feature_names=np.array([str(c) for c in EDGE_COLS], dtype="U40"))
    summary_rows.append({"bucket": bname, "graphs": len(ids), "nodes": len(nidx), "edges_undirected": int(sum(ecount[s] for s in ids)),
                         "mean_nodes": round(len(nidx) / len(ids), 2), "label1_events": int(node_label_all[nidx].sum())})
    print("saved", SPLIT_DIR / f"graphs_b1_{bname}_v6v2.npz")
bucket_summary = pd.DataFrame(summary_rows).set_index("bucket")
display(bucket_summary)

## Step 9 — Save split metadata and run internal integrity checks

Export the split mapping/manifest and verification report to the designated folders. Validate expected graph counts and bucket composition. **This notebook does not compare against the older B1 files** — that comparison belongs to the separate audit, as requested.

In [ ]:
import json
split=pd.concat([pd.DataFrame({'scenario_id':ids,'bucket':name}) for name,ids in buckets.items()],ignore_index=True)
split=split.merge(summ[['scenario_id','type','n_events','n_label1']],on='scenario_id').sort_values('scenario_id')
split.to_csv(SPLIT_DIR/'split_b1_v6v2.csv',index=False)
meta_export=meta.reset_index()[['scenario_id','split','is_attack','scenario_type','attack_type','n_events_total','group','group_size','stratum']]
meta_export.to_csv(REPORT_DIR/'split_manifest_approach_b_v6v2.csv',index=False)
expected={'train':(8946,0),'validation':(1278,556),'test_normal':(1112,0),'test_attack':(1444,1444)}
all_ids=[];result={}
for key,(n,na) in expected.items():
    f=SPLIT_DIR/f'graphs_b1_{key}_v6v2.npz'
    with np.load(f,allow_pickle=False) as z:
        ids=z['scenario_id']; x=z['x']; ei=z['edge_index'];ea=z['edge_attr']; nptr=z['node_ptr'];eptr=z['edge_ptr']; y=z['scenario_is_attack']
        assert len(ids)==n and int(y.sum())==na and x.shape[1]==92 and ea.shape[1]==14
        assert len(nptr)==n+1 and len(eptr)==n+1 and np.isfinite(x).all() and np.isfinite(ea).all()
        assert len(x)==nptr[-1] and len(ea)==eptr[-1] and ei.shape[1]==len(ea)
        assert np.array_equal(ids,np.sort(ids))
        assert all(int(ei[:,eptr[j]:eptr[j+1]].max())<int(nptr[j+1]-nptr[j]) for j in range(n))
        all_ids.extend(map(int,ids))
        result[key]={'graphs':n,'attack_graphs':na,'nodes':int(len(x)),'directed_edges':int(len(ea))}
assert len(all_ids)==len(set(all_ids))==12780
assert set(all_ids)==set(map(int,summ['scenario_id']))
report={'source':'Stage 3b + original Host/IoT logs','seed':SEED,'target':TARGET,'actual':result,
        'features':{'node':92,'edge':14},'scenario_overlap':0,'validation_contains_normal_and_attack':True,
        'previous_B1_files_compared':False}
(REPORT_DIR/'split_verification_approach_b_v6v2.json').write_text(json.dumps(report,indent=2),encoding='utf8')
print('VALIDATED — four B1 files, unique IDs, mixed validation, graph-local edges.')
for key,details in result.items():print(key,details)
print('Output:',OUT_ROOT)


## Final output checklist

The output location is `MyDrive/GNN/Stage 4/Approach B/`:

| Folder | Expected artifacts |
|---|---|
| `graph_data/` | Node features, edges, labels, scenario info, feature names (CSV) |
| `splits/` | 4 B1 NPZ graph files, split manifest CSV, scaling CSV |
| `reports/` | Split verification and scenario metadata reports |

**Validation before model selection:** inspect the final cell's split counts and errors. Keep the original model-training B1 files unchanged. Dataset construction and split reproducibility are distinct from the broader model/data leakage audit.